# 33 孟加拉国 DIA 冻结队列公开实测谱库匹配

本 Notebook 只处理 Notebook 32 已冻结的 400 个跨极性簇（464 条 Pos/Neg 谱图），在本地清洗并审计通过的 GNPS、MassBank 和 MoNA 实测谱库中进行身份揭示。Notebook 31 的排名和 Notebook 32 的队列均不得因匹配结果而改变。

预先固定的主匹配规则：

- 离子模式一致；
- 规范化加合物一致；
- 前体离子误差不超过 20 ppm；
- 碎片容差 0.02 Da；
- sqrt-intensity modified cosine 不低于 0.70；
- 匹配碎片峰不少于 6。

高置信候选要求 cosine ≥ 0.85 且匹配峰 ≥ 8。最佳和次佳不同连接结构的分数差 ≤ 0.05 时标记为歧义。未匹配特征仅代表尚未注释，绝不视为阴性。

> DIA 去卷积谱可能含共碎裂信号，且没有标准品 RT；因此结果只能称为实测谱库连接结构候选，不能称为 Level 1 确证。内核必须使用 `env_ms`。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'pbt_msms').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
assert Path(sys.prefix).name.lower() == 'env_ms', f'请切换到 env_ms 内核；当前为 {sys.executable}'
print('ROOT =', ROOT)
print('Python =', sys.executable)


In [ ]:
from pbt_msms.bangladesh_dia_library import paths, sha256_file
p = paths(ROOT)
stage32 = json.loads(p['cohort_manifest'].read_text(encoding='utf-8'))
library_summary = json.loads(p['public_summary'].read_text(encoding='utf-8'))
assert stage32['status'] == 'IDENTITY_REVEAL_COHORT_FROZEN'
assert stage32['n_clusters_selected'] == 400
assert stage32['n_feature_members_selected'] > 0
assert library_summary['status'] == 'PASS'
assert library_summary['export']['record_count'] == 1702963
print('Notebook 32 frozen gate: PASS')
print('Public library cleaning gate: PASS')
print('Public spectra:', library_summary['export']['record_count'])


## 1. 建立冻结候选谱库

下面的单元格会完整扫描本地 1,702,963 条谱库元数据，但只加载落入冻结谱图离子模式、加合物和 20 ppm 前体窗口的实测谱图。第一次运行通常是本 Notebook 最耗时的准备步骤。

In [ ]:
from pbt_msms.bangladesh_dia_library import prepare
preparation = prepare(ROOT)
print(json.dumps(preparation, ensure_ascii=False, indent=2))
assert preparation['status'] == 'LIBRARY_CANDIDATES_PREPARED'
assert preparation['counts']['frozen_clusters'] == 400
assert preparation['counts']['case_spectra'] == stage32['n_feature_members_selected']


## 2. Modified-cosine 检索与簇级竞争

对质量窗口中的参考谱图计算 modified cosine；同一连接结构保留最佳参考谱，再在每条特征和每个跨极性簇内比较不同身份。

In [ ]:
from pbt_msms.bangladesh_dia_library import match
manifest = match(ROOT)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status'] == 'PUBLIC_LIBRARY_MATCHING_COMPLETE'
assert manifest['counts']['frozen_clusters'] == 400


In [ ]:
import pandas as pd
OUT = p['out']
clusters = pd.read_csv(OUT / '33_cluster_library_match_summary.csv')
features = pd.read_csv(OUT / '33_feature_best_library_match.csv.gz')
unresolved = pd.read_csv(OUT / '33_clusters_without_nonambiguous_library_match.csv')
audit = pd.read_csv(OUT / '33_cohort_match_audit.csv')
assert len(clusters) == 400 and clusters['cross_polarity_cluster'].is_unique
assert len(features) == manifest['counts']['case_spectra'] and features['spectrum_id'].is_unique
assert len(unresolved) == manifest['counts']['clusters_without_nonambiguous_library_match']
assert audit['n_clusters'].sum() == 400
assert audit['nonambiguous_library_match'].sum() + audit['without_nonambiguous_library_match'].sum() == 400
for filename, details in manifest['artifacts'].items():
    assert sha256_file(OUT / filename) == details['sha256'], f'哈希不一致：{filename}'
display(audit)
display(clusters.loc[clusters['is_nonambiguous_library_match'].fillna(False).astype(bool), [
    'cohort', 'cluster_rank', 'cross_polarity_cluster', 'inchikey_2d', 'canonical_smiles',
    'modified_cosine', 'matched_peaks', 'precursor_ppm_error', 'supporting_modes',
    'reference_source_library', 'library_match_status'
]].sort_values(['cohort', 'cluster_rank']).head(30))
print('400 个冻结簇完整、匹配阈值合规、输出哈希：全部 PASS')


## 完成判据与下一步

最后一个单元格必须显示：`400 个冻结簇完整、匹配阈值合规、输出哈希：全部 PASS`。

输出目录：`external_data/bangladesh_surface_water/06_revealed_evaluation/33_public_library_matching_v1`。

`33_clusters_without_nonambiguous_library_match.csv` 单独记录无主阈值匹配或匹配歧义的簇；Notebook 34 为避免差异性验证，将对全部 400 个身份盲态冻结簇执行同一套 SIRIUS 注释。